Import

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import itertools
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from gensim.models import Word2Vec

import mlflow

mlflow.set_experiment("")  

labelaa='Single_Label'
df = pd.read_pickle('springer_dataframe_5_categories.p')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[labelaa],
    random_state=40
)

test_df, val_df = train_test_split(
    test_df,
    test_size=0.5,
    stratify=test_df[labelaa],
    random_state=40
)

def compute_prototypes(embedding_net, source_df, labels, vector_col, device):
    prototypes = {}
    with torch.no_grad():
        for label in labels:
            class_df = source_df[source_df[labelaa] == label]
            class_vectors = torch.tensor(
                np.stack(class_df[vector_col].values), dtype=torch.float32
            ).to(device)
            class_embeddings = embedding_net(class_vectors)
            prototypes[label] = torch.mean(class_embeddings, dim=0)
    return prototypes
 
 
def compute_accuracy(embedding_net, prototypes, eval_df, vector_col, device):
    eval_vectors_np = np.stack(eval_df[vector_col].values)
    eval_vectors = torch.tensor(eval_vectors_np, dtype=torch.float32).to(device)
    eval_labels = eval_df[labelaa].values
 
    with torch.no_grad():
        eval_embeddings = embedding_net(eval_vectors)
 
    correct = 0
    for i in range(len(eval_labels)):
        embed = eval_embeddings[i]
        true_label = eval_labels[i]
 
        best_label = None
        smallest_distance = float('inf')
        for label, prototype in prototypes.items():
            sim = F.cosine_similarity(embed.unsqueeze(0), prototype.unsqueeze(0))
            dist = 1.0 - sim.item()
            if dist < smallest_distance:
                smallest_distance = dist
                best_label = label
 
        if best_label == true_label:
            correct += 1
 
    return (correct / len(eval_labels)) * 100
 
 

Word2vec

In [ ]:
w2v_df = pd.read_pickle('springer_dataframe_26_categories.p')
w2v_train_tokens = [str(text).split() for text in w2v_df['toc']]

EMBEDDING_DIM = 300 

w2v_model = Word2Vec(
    w2v_train_tokens,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=5,
    workers=4,
    epochs=15
)


def tokenize(text):
    return re.findall(r'\b[a-zA-Z]+\b', str(text).lower())


def doc_to_vector(text, model, embedding_dim=EMBEDDING_DIM):
    words = tokenize(text)
    vectors = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(embedding_dim)



train_df['vector'] = train_df['toc'].apply(lambda text: doc_to_vector(text, w2v_model))
val_df['vector'] = val_df['toc'].apply(lambda text: doc_to_vector(text, w2v_model))
test_df['vector'] = test_df['toc'].apply(lambda text: doc_to_vector(text, w2v_model))

labels = train_df[labelaa].unique()


TFIDF

In [ ]:
EMBEDDING_DIM = 5000

tfidf = TfidfVectorizer(max_features=EMBEDDING_DIM, stop_words='english')

tfidf_matrix = tfidf.fit_transform(train_df['toc'])
train_df['vector'] = [np.array(vec).flatten() for vec in tfidf_matrix.toarray()]

val_tfidf_matrix = tfidf.transform(val_df['toc'])
val_df['vector'] = [np.array(vec).flatten() for vec in val_tfidf_matrix.toarray()]

labels = train_df[labelaa].unique()

BERT

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer  


bert_model = SentenceTransformer('all-mpnet-base-v2')
train_vectors = bert_model.encode(train_df['toc'].astype(str).tolist(), show_progress_bar=True)
val_vectors = bert_model.encode(val_df['toc'].astype(str).tolist(), show_progress_bar=True)
test_vectors = bert_model.encode(test_df['toc'].astype(str).tolist(), show_progress_bar=True)

train_df['vector'] = list(train_vectors)
val_df['vector'] = list(val_vectors)
test_df['vector'] = list(test_vectors)


Parovi

In [ ]:
def build_pairs(source_df, labels):
    pairs_list = []
    for label1, label2 in itertools.combinations_with_replacement(labels, 2):
        df_class1 = source_df[source_df[labelaa] == label1]
        df_class2 = source_df[source_df[labelaa] == label2]

        N = min(len(df_class1), len(df_class2))
        if N == 0:
            continue

        left_samples = df_class1.sample(n=N, replace=True).reset_index(drop=True)
        right_samples = df_class2.sample(n=N, replace=True).reset_index(drop=True)

        is_same = 1 if label1 == label2 else 0

        for i in range(N):
            pairs_list.append({
                'input_left': left_samples.iloc[i]['vector'],
                'input_right': right_samples.iloc[i]['vector'],
                'label_left': label1,
                'label_right': label2,
                'is_same': is_same
            })
    return pairs_list


train_pairs_df = pd.DataFrame(build_pairs(train_df, labels))
train_pairs_df = train_pairs_df.sample(frac=1, random_state=42).reset_index(drop=True)

val_pairs_df = pd.DataFrame(build_pairs(val_df, labels))
val_pairs_df = val_pairs_df.sample(frac=1, random_state=42).reset_index(drop=True)



class EmbeddingNetwork(nn.Module):
    def __init__(self, input_dim=EMBEDDING_DIM, embedding_dim=128):
        super(EmbeddingNetwork, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        return self.fc(x)


class SiameseNetwork(nn.Module):
    def __init__(self, embedding_net):
        super(SiameseNetwork, self).__init__()
        self.embedding_net = embedding_net

    def forward(self, x_left, x_right):
        output_left = self.embedding_net(x_left)
        output_right = self.embedding_net(x_right)
        return output_left, output_right


class SiameseDataset(Dataset):
    def __init__(self, pairs_df):
        self.x_left = np.stack(pairs_df['input_left'].values)
        self.x_right = np.stack(pairs_df['input_right'].values)
        self.labels = pairs_df['is_same'].values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.x_left[idx], dtype=torch.float32),
            torch.tensor(self.x_right[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )


train_dataset = SiameseDataset(train_pairs_df)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_dataset = SiameseDataset(val_pairs_df)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class CosineContrastiveLoss(nn.Module):
    def __init__(self, margin=1):
        super(CosineContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output_left, output_right, target):
       
        cosine_sim = F.cosine_similarity(output_left, output_right)
        
        cosine_dist = 1 - cosine_sim
        
        loss_same = target * torch.pow(cosine_dist, 2)

        loss_diff = (1 - target) * torch.pow(torch.clamp(self.margin - cosine_dist, min=0.0), 2)
        
        return torch.mean(loss_same + loss_diff)

emb_net = EmbeddingNetwork(input_dim=EMBEDDING_DIM, embedding_dim=128)
model = SiameseNetwork(emb_net).to(device)

NUM_EPOCHS=20
LR=0.005
criterion = CosineContrastiveLoss(margin=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)




with mlflow.start_run():
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", NUM_EPOCHS)
 
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
 
        for batch_left, batch_right, batch_labels in train_loader:
            batch_left = batch_left.to(device)
            batch_right = batch_right.to(device)
            batch_labels = batch_labels.to(device)
 
            optimizer.zero_grad()
 
            out_left, out_right = model(batch_left, batch_right)
 
            loss = criterion(out_left, out_right, batch_labels)
            loss.backward()
            optimizer.step()
 
            running_loss += loss.item() * batch_left.size(0)
 
        epoch_loss = running_loss / len(train_loader.dataset)
 
        model.eval()
        val_running_loss = 0.0
 
        with torch.no_grad():
            for val_left, val_right, val_labels in val_loader:
                val_left = val_left.to(device)
                val_right = val_right.to(device)
                val_labels = val_labels.to(device)
 
                val_out_left, val_out_right = model(val_left, val_right)
                val_loss = criterion(val_out_left, val_out_right, val_labels)
 
                val_running_loss += val_loss.item() * val_left.size(0)
 
        epoch_val_loss = val_running_loss / len(val_loader.dataset)
 
        prototypes = compute_prototypes(emb_net, train_df, labels, 'vector', device)
        val_accuracy = compute_accuracy(emb_net, prototypes, val_df, 'vector', device)
 
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        mlflow.log_metric("val_loss", epoch_val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)
 
        print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] -> Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%")            
    model.eval()
    final_prototypes = compute_prototypes(emb_net, train_df, labels, device)
    test_accuracy = compute_accuracy(emb_net, final_prototypes, test_df, device)

    mlflow.log_metric("final_accuracy", test_accuracy)

    print("-" * 40)
    print("Evaluation Results:")
    print(f"Total Test Samples: {len(test_df)}")
    print(f"Final Model Accuracy: {test_accuracy:.2f}%")
 

Trojke

In [ ]:
triplets_list = []

def build_triplets(source_df, labels):
    triplets_list = []
    for label1, label2 in itertools.combinations(labels, 2):

        df_class1 = source_df[source_df['LCSH_Label'] == label1]
        df_class2 = source_df[source_df['LCSH_Label'] == label2]
        N=min(len(df_class1), len(df_class2))
        if len(df_class1) == 0 or len(df_class2) == 0:
            continue

        anchor_samples = df_class1.sample(n=N, replace=True).reset_index(drop=True)
        positive_samples = df_class1.sample(n=N, replace=True).reset_index(drop=True)
        negative_samples = df_class2.sample(n=N, replace=True).reset_index(drop=True)

        for i in range(N):
            triplets_list.append({
                'input_anchor': anchor_samples.iloc[i]['vector'],
                'input_positive': positive_samples.iloc[i]['vector'],
                'input_negative': negative_samples.iloc[i]['vector']
            })
    return triplets_list


train_triplets_df = pd.DataFrame(build_triplets(train_df, labels))
train_triplets_df = train_triplets_df.sample(frac=1, random_state=42).reset_index(drop=True)


val_triplets_df = pd.DataFrame(build_triplets(val_df, labels))
val_triplets_df = val_triplets_df.sample(frac=1, random_state=42).reset_index(drop=True)


class TripletDataset(Dataset):
    def __init__(self, triplets_df):
        self.anchor = np.stack(triplets_df['input_anchor'].values)
        self.positive = np.stack(triplets_df['input_positive'].values)
        self.negative = np.stack(triplets_df['input_negative'].values)

    def __len__(self):
        return len(self.anchor)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.anchor[idx], dtype=torch.float32),
            torch.tensor(self.positive[idx], dtype=torch.float32),
            torch.tensor(self.negative[idx], dtype=torch.float32)
        )

train_dataset = TripletDataset(train_triplets_df)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_dataset = TripletDataset(val_triplets_df)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)

class EmbeddingNetwork(nn.Module):
    def __init__(self, input_dim=EMBEDDING_DIM, embedding_dim=128):
        super(EmbeddingNetwork, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        return self.fc(x)


class TripletNetwork(nn.Module):
    def __init__(self, embedding_net):
        super(TripletNetwork, self).__init__()
        self.embedding_net = embedding_net

    def forward(self, x_anchor, x_positive, x_negative):
        out_anchor = self.embedding_net(x_anchor)
        out_positive = self.embedding_net(x_positive)
        out_negative = self.embedding_net(x_negative)
        return out_anchor, out_positive, out_negative

class CosineTripletLoss(nn.Module):
    def __init__(self, margin=1.2):
        super(CosineTripletLoss, self).__init__()
        self.margin = margin

    def forward(self, out_anchor, out_positive, out_negative):

        sim_pos = F.cosine_similarity(out_anchor, out_positive)
        sim_neg = F.cosine_similarity(out_anchor, out_negative)

        dist_pos = 1.0 - sim_pos
        dist_neg = 1.0 - sim_neg

        loss = torch.clamp(dist_pos - dist_neg + self.margin, min=0.0)
        return torch.mean(loss)


emb_net = EmbeddingNetwork(input_dim=EMBEDDING_DIM, embedding_dim=64)
model = TripletNetwork(emb_net).to(device)


LR = 0.001
criterion = CosineTripletLoss(margin=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
NUM_EPOCHS = 20


with mlflow.start_run():
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", NUM_EPOCHS)

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for batch_anchor, batch_positive, batch_negative in train_loader:
            batch_anchor = batch_anchor.to(device)
            batch_positive = batch_positive.to(device)
            batch_negative = batch_negative.to(device)

            optimizer.zero_grad()

            out_anchor, out_positive, out_negative = model(batch_anchor, batch_positive, batch_negative)

            loss = criterion(out_anchor, out_positive, out_negative)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * batch_anchor.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()

        running_val_loss = 0.0
        with torch.no_grad():
            for batch_anchor, batch_positive, batch_negative in val_loader:
                batch_anchor, batch_positive, batch_negative = batch_anchor.to(device), batch_positive.to(device), batch_negative.to(device)

                out_anchor, out_positive, out_negative = model(batch_anchor, batch_positive, batch_negative)
                loss = criterion(out_anchor, out_positive, out_negative)

                running_val_loss += loss.item() * batch_anchor.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)


        prototypes = compute_prototypes(emb_net, train_df, labels, device)
        val_accuracy = compute_accuracy(emb_net, prototypes, val_df, device)

        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        mlflow.log_metric("val_loss", epoch_val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)

        print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] -> Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%")

model.eval()
final_prototypes = compute_prototypes(emb_net, train_df, labels, device)

test_tfidf_matrix = tfidf.transform(test_df['toc'])
test_df['vector'] = [np.array(vec).flatten() for vec in test_tfidf_matrix.toarray()]

test_accuracy = compute_accuracy(emb_net, final_prototypes, test_df, device)

print("-" * 40)
print(f"Evaluation Results:")
print(f"Total Test Samples: {len(test_df)}")
print(f"Final Model Accuracy: {test_accuracy:.2f}%")
